In [ ]:
# ===== CONFIGURE HERE (Change based on your setup) =====
# For Google Colab:
# BASE_URL = "/content/drive/MyDrive/experiments"

# For Local (Windows):
BASE_URL = "D:/MPHIL_CODES/MPHIL_MAIN_REPO/experiments"

# For Local (Mac/Linux):
# BASE_URL = "/Users/yourname/experiments"

from pathlib import Path
BASE_PATH = Path(BASE_URL)
ARCADE_PATH = BASE_PATH / 'datasets' / 'ARCADE'

# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    pass

# cGAN Training & Testing - Colab (FIXED)

Train and test the UCNet cGAN model on ARCADE dataset

**All files in:** `/content/drive/MyDrive/MPhil Research/Datasets/`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision numpy pillow tqdm matplotlib pandas

## 3. Setup Paths & Extract Dataset

In [ ]:
import sys
import os
import shutil
import zipfile
from pathlib import Path

# Base directory where all files are
DATASETS_DIR = Path('/content/drive/MyDrive/MPhil Research/Datasets')

print(f"Working directory: {DATASETS_DIR}")
print(f"\nContents of Datasets folder:")
for item in sorted(os.listdir(DATASETS_DIR)):
    item_path = DATASETS_DIR / item
    if item_path.is_dir():
        print(f"  📁 {item}/")
    else:
        print(f"  📄 {item}")

## 4. Extract ARCADE Dataset (if needed)

In [ ]:
# Check if ARCADE is already extracted
arcade_path = DATASETS_DIR / 'ARCADE'
arcade_zip = DATASETS_DIR / 'ARCADE.zip'

if arcade_path.exists():
    print(f"✓ ARCADE folder already exists at {arcade_path}")
else:
    if arcade_zip.exists():
        print(f"Extracting ARCADE.zip...")
        with zipfile.ZipFile(arcade_zip, 'r') as zip_ref:
            zip_ref.extractall(DATASETS_DIR)
        print(f"✓ ARCADE extracted")
    else:
        print(f"✗ ERROR: ARCADE.zip not found in {DATASETS_DIR}")

# Verify ARCADE structure
if arcade_path.exists():
    print(f"\nARCADE structure:")
    for item in os.listdir(arcade_path):
        print(f"  {item}")

## 5. Setup ucnet Module

In [ ]:
# Clear any cached imports
modules_to_remove = [m for m in list(sys.modules.keys()) if 'data' in m or 'models' in m or 'utils' in m]
for module in modules_to_remove:
    del sys.modules[module]

# Clear sys.path
sys.path = [p for p in sys.path if 'ucnet' not in p]

# Copy ucnet to /content/ for faster access
ucnet_drive = DATASETS_DIR / 'ucnet'
ucnet_content = Path('/content/ucnet')

print(f"Checking ucnet at: {ucnet_drive}")
print(f"ucnet exists: {ucnet_drive.exists()}")

if ucnet_drive.exists():
    if ucnet_content.exists():
        shutil.rmtree(ucnet_content)
    
    print(f"Copying ucnet to {ucnet_content}...")
    shutil.copytree(ucnet_drive, ucnet_content)
    print(f"✓ ucnet copied")
else:
    print(f"✗ ERROR: ucnet folder not found at {ucnet_drive}")
    print(f"Please upload ucnet folder to {DATASETS_DIR}")

# Add to path
sys.path.insert(0, str(ucnet_content))
print(f"\nAdded to sys.path: {ucnet_content}")

## 6. Test Imports

In [ ]:
import traceback

print("Testing imports:\n")

try:
    from data.arcade_dataset import ArcadeSegmentDataset
    print("✓ ArcadeSegmentDataset")
except Exception as e:
    print(f"✗ ArcadeSegmentDataset: {e}")
    traceback.print_exc()

try:
    from models.generator import ImprovedUNetGenerator
    print("✓ ImprovedUNetGenerator")
except Exception as e:
    print(f"✗ ImprovedUNetGenerator: {e}")
    traceback.print_exc()

try:
    from utils.metrics import SegmentationMeter
    print("✓ SegmentationMeter")
except Exception as e:
    print(f"✗ SegmentationMeter: {e}")
    traceback.print_exc()

print("\n✓ All imports successful!")

## 7. Load Datasets

In [ ]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm
import pandas as pd

# Paths
TRAIN_IMAGES = arcade_path / 'stenosis' / 'train' / 'images'
TRAIN_ANN = arcade_path / 'stenosis' / 'train' / 'annotations' / 'train.json'
VAL_IMAGES = arcade_path / 'stenosis' / 'val' / 'images'
VAL_ANN = arcade_path / 'stenosis' / 'val' / 'annotations' / 'val.json'

# Output directory
OUTPUT_DIR = DATASETS_DIR / 'cGAN_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

print(f"Train images: {TRAIN_IMAGES}")
print(f"Train annotations: {TRAIN_ANN}")
print(f"Output directory: {OUTPUT_DIR}")

# Verify paths
print(f"\n✓ Train images exist: {TRAIN_IMAGES.exists()}")
print(f"✓ Train annotations exist: {TRAIN_ANN.exists()}")
print(f"✓ Val images exist: {VAL_IMAGES.exists()}")
print(f"✓ Val annotations exist: {VAL_ANN.exists()}")

## 8. Load Training & Validation Datasets

In [ ]:
print("Loading datasets...\n")

train_dataset = ArcadeSegmentDataset(
    image_dir=str(TRAIN_IMAGES),
    ann_file=str(TRAIN_ANN),
    img_size=512,
    augment=True
)

val_dataset = ArcadeSegmentDataset(
    image_dir=str(VAL_IMAGES),
    ann_file=str(VAL_ANN),
    img_size=512,
    augment=False
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Number of classes: {train_dataset.num_classes}")

# Save metadata
cat2class = train_dataset.cat2class
class_names = train_dataset.class_names
num_classes = train_dataset.num_classes

## 9. Configuration (Paper Settings)

In [ ]:
CONFIG = {
    'epochs': 400,  # Paper uses 400 epochs
    'batch_size': 4,  # Paper uses batch size 4
    'learning_rate': 2e-4,  # Paper uses 2e-4
    'num_classes': num_classes,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers': 0,
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

device = torch.device(CONFIG['device'])
print(f"\n✓ Using device: {device}")

## 10. Create Data Loaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers']
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=CONFIG['num_workers']
)

print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val batches per epoch: {len(val_loader)}")

## 11. Initialize Model

In [ ]:
G = ImprovedUNetGenerator(
    in_channels=3,
    out_channels=CONFIG['num_classes']
).to(device)

total_params = sum(p.numel() for p in G.parameters())
print(f"✓ Generator initialized")
print(f"  Total parameters: {total_params:,}")

## 12. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(G.parameters(), lr=CONFIG['learning_rate'])

best_val_f1 = 0.0
training_history = []

print(f"Starting training for {CONFIG['epochs']} epochs...\n")

for epoch in range(CONFIG['epochs']):
    # ========== TRAINING ==========
    G.train()
    train_loss = 0.0
    
    for x, target in train_loader:
        x, target = x.to(device), target.to(device)
        
        optimizer.zero_grad()
        logits = G(x)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    # ========== VALIDATION WITH METRICS ==========
    G.eval()
    val_loss = 0.0
    meter = SegmentationMeter(CONFIG['num_classes'], class_names=class_names)
    
    with torch.no_grad():
        for x, target in val_loader:
            x, target = x.to(device), target.to(device)
            logits = G(x)
            
            # Loss
            loss = criterion(logits, target)
            val_loss += loss.item()
            
            # Metrics
            meter.update(logits, target)
    
    val_loss /= len(val_loader)
    
    # Compute metrics
    metrics, per_class_metrics = meter.compute()
    val_f1 = metrics['f1']
    val_accuracy = metrics['accuracy']
    val_iou = metrics['iou']
    
    training_history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_f1': val_f1,
        'val_accuracy': val_accuracy,
        'val_iou': val_iou
    })
    
    # ========== SAVE BEST CHECKPOINT (by F1) ==========
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        checkpoint = {
            'epoch': epoch + 1,
            'G': G.state_dict(),
            'optimizer': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_f1': val_f1,
            'val_accuracy': val_accuracy,
            'val_iou': val_iou,
            'num_classes': CONFIG['num_classes'],
            'cat2class': cat2class,
            'class_names': class_names
        }
        torch.save(checkpoint, CHECKPOINT_DIR / 'ucnet_best.pth')
        best_epoch = epoch + 1
    
    # ========== PRINT PROGRESS ==========
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{CONFIG['epochs']} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"F1: {val_f1:.4f} | "
              f"Acc: {val_accuracy:.4f} | "
              f"IoU: {val_iou:.4f}")

print(f"\n{'='*80}")
print(f"✓ TRAINING COMPLETE!")
print(f"{'='*80}")
print(f"Best model saved at epoch: {best_epoch}")
print(f"Best F1 Score: {best_val_f1:.4f}")
print(f"Model path: {CHECKPOINT_DIR / 'ucnet_best.pth'}")

## 13. Save Training History

In [ ]:
import json
import matplotlib.pyplot as plt

# Save history
with open(OUTPUT_DIR / 'training_history.json', 'w') as f:
    json.dump(training_history, f, indent=2)

# Extract data
epochs = [h['epoch'] for h in training_history]
train_losses = [h['train_loss'] for h in training_history]
val_losses = [h['val_loss'] for h in training_history]
val_f1s = [h['val_f1'] for h in training_history]
val_accs = [h['val_accuracy'] for h in training_history]
val_ious = [h['val_iou'] for h in training_history]

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training History & Metrics', fontsize=16, fontweight='bold')

# Plot 1: Loss
axes[0, 0].plot(epochs, train_losses, label='Train Loss', alpha=0.7, linewidth=2)
axes[0, 0].plot(epochs, val_losses, label='Val Loss', alpha=0.7, linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: F1 Score
axes[0, 1].plot(epochs, val_f1s, label='F1 Score', color='green', alpha=0.7, linewidth=2)
axes[0, 1].axhline(y=max(val_f1s), color='red', linestyle='--', alpha=0.5, label=f'Max: {max(val_f1s):.4f}')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].set_title('F1 Score over Epochs')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 1])

# Plot 3: Accuracy
axes[1, 0].plot(epochs, val_accs, label='Accuracy', color='orange', alpha=0.7, linewidth=2)
axes[1, 0].axhline(y=max(val_accs), color='red', linestyle='--', alpha=0.5, label=f'Max: {max(val_accs):.4f}')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('Accuracy over Epochs')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1])

# Plot 4: IoU
axes[1, 1].plot(epochs, val_ious, label='IoU Score', color='purple', alpha=0.7, linewidth=2)
axes[1, 1].axhline(y=max(val_ious), color='red', linestyle='--', alpha=0.5, label=f'Max: {max(val_ious):.4f}')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('IoU Score')
axes[1, 1].set_title('IoU Score over Epochs')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Training history saved with all metrics")

## 14. Load Best Model for Testing

In [ ]:
# Load best checkpoint
checkpoint = torch.load(CHECKPOINT_DIR / 'ucnet_best.pth', map_location=device)

# Create model
G_test = ImprovedUNetGenerator(
    in_channels=3,
    out_channels=checkpoint['num_classes']
).to(device)

G_test.load_state_dict(checkpoint['G'])
G_test.eval()

print(f"✓ Best model loaded")
print(f"  Epoch: {checkpoint['epoch']}")
print(f"  Val Loss: {checkpoint['val_loss']:.4f}")

## 15. Test on Validation Set

In [ ]:
from utils.metrics import SegmentationMeter

print("Testing on validation set...")
meter = SegmentationMeter(checkpoint['num_classes'], class_names=checkpoint['class_names'])

with torch.no_grad():
    for batch_idx, (x, target) in enumerate(val_loader):
        if (batch_idx + 1) % 50 == 0:
            print(f"  Processed {batch_idx + 1}/{len(val_loader)}")
        
        x, target = x.to(device), target.to(device)
        logits = G_test(x)
        meter.update(logits, target)

# Compute metrics
metrics, per_class_metrics = meter.compute()

print("\n" + "="*80)
print("VALIDATION TEST RESULTS")
print("="*80)
print("\nOverall Metrics:")
for key, value in metrics.items():
    print(f"  {key:20s}: {value:.4f}")

## 16. Sample Predictions

In [ ]:
# Show sample predictions
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
fig.suptitle('Sample Predictions', fontsize=14, fontweight='bold')

test_loader_demo = DataLoader(val_dataset, batch_size=1, shuffle=True, num_workers=0)

with torch.no_grad():
    for idx, (x, target) in enumerate(test_loader_demo):
        if idx >= 3:
            break
        
        x, target = x.to(device), target.to(device)
        logits = G_test(x)
        pred = torch.argmax(logits, dim=1)[0].cpu().numpy()
        
        # Input
        axes[idx, 0].imshow(x[0, 0].cpu().numpy(), cmap='gray')
        axes[idx, 0].set_title(f'Input {idx+1}')
        axes[idx, 0].axis('off')
        
        # Ground Truth
        axes[idx, 1].imshow(target[0].cpu().numpy(), cmap='tab20')
        axes[idx, 1].set_title(f'Ground Truth {idx+1}')
        axes[idx, 1].axis('off')
        
        # Prediction
        axes[idx, 2].imshow(pred, cmap='tab20')
        axes[idx, 2].set_title(f'Prediction {idx+1}')
        axes[idx, 2].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Sample predictions saved")

## 17. Final Summary

In [ ]:
print("\n" + "="*80)
print("TRAINING & TESTING COMPLETE")
print("="*80)

print(f"\nConfiguration:")
print(f"  Epochs: {CONFIG['epochs']}")
print(f"  Batch Size: {CONFIG['batch_size']}")
print(f"  Learning Rate: {CONFIG['learning_rate']}")
print(f"  Device: {CONFIG['device']}")

print(f"\nResults:")
print(f"  Best Validation Loss: {best_val_loss:.4f}")
print(f"  F1 Score: {metrics['f1']:.4f}")
print(f"  IoU Score: {metrics['iou']:.4f}")
print(f"  Accuracy: {metrics['accuracy']:.4f}")
print(f"  Sensitivity: {metrics['sensitivity']:.4f}")
print(f"  Specificity: {metrics['specificity']:.4f}")

print(f"\nOutput Files:")
print(f"  Model: {CHECKPOINT_DIR / 'ucnet_best.pth'}")
print(f"  Training history: {OUTPUT_DIR / 'training_history.json'}")
print(f"  Training plot: {OUTPUT_DIR / 'training_history.png'}")
print(f"  Predictions: {OUTPUT_DIR / 'sample_predictions.png'}")

print(f"\n✓ All outputs saved to: {OUTPUT_DIR}")